# Notebook 11 — SciPy: Optimization

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 11 of 20  
**Prerequisites:** Notebook 10  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Optimization is the engine of computational biology: maximum likelihood estimation
for phylogenetics, parameter fitting for kinetic models, loss minimization for
neural networks. `scipy.optimize` provides production-grade solvers for all of these.
Understanding when each solver is appropriate is more important than memorizing the API.

---
## Step 2 — Intuition

Minimizing a function means finding the input value that produces the lowest output.
Gradient-based solvers follow the slope downhill. Gradient-free solvers explore the
space using evaluations alone. Constrained solvers enforce boundaries.

Choosing the wrong solver is the most common mistake: using a gradient-free solver
on a smooth convex function wastes thousands of evaluations that a quasi-Newton
solver would solve in 10.

---
## Step 3 — Biological Background

**Michaelis-Menten kinetics:**

The reaction rate $v$ as a function of substrate concentration $[S]$:

$$v = \frac{V_{\max} [S]}{K_m + [S]}$$

where $V_{\max}$ is the maximum rate and $K_m$ is the substrate concentration at
half-maximum rate. Given experimental $(\{[S]_i\}, \{v_i\})$ data, fitting the
model means finding $(V_{\max}, K_m)$ that minimizes the sum of squared residuals.

This is a two-parameter, nonlinear least-squares problem — exactly what `curve_fit` solves.

---
## Step 4 — Mathematical Explanation

**Nonlinear least squares:**

$$\arg\min_{\theta} \sum_i \left(y_i - f(x_i; \theta)\right)^2$$

For smooth $f$, the Levenberg-Marquardt algorithm is the default:
it interpolates between gradient descent (far from the minimum) and Gauss-Newton
(near the minimum).

**Log-likelihood maximization** is equivalent to minimizing the negative log-likelihood —
same mathematical framework, different biological context.

---
## Step 5 — Computational Explanation

| Task | Function | Notes |
|------|----------|-------|
| Nonlinear curve fitting | `scipy.optimize.curve_fit` | Returns params + covariance |
| General minimization | `scipy.optimize.minimize` | Many methods; L-BFGS-B for smooth |
| Root finding | `scipy.optimize.brentq` | Finds f(x)=0 on an interval |
| Bounded minimization (1-D) | `scipy.optimize.minimize_scalar` | Simpler than `minimize` for scalar |
| Differential evolution | `scipy.optimize.differential_evolution` | Global, gradient-free |

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
from scipy.optimize import curve_fit, minimize, brentq
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

In [ ]:
# Cell 6.1 — Michaelis-Menten curve fitting
def michaelis_menten(S: np.ndarray, Vmax: float, Km: float) -> np.ndarray:
    """Michaelis-Menten kinetic model."""
    return Vmax * S / (Km + S)

# True parameters
TRUE_VMAX, TRUE_KM = 10.0, 2.5

# Synthetic experimental data: substrate concentrations in µM
S_data = np.array([0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0])
v_true = michaelis_menten(S_data, TRUE_VMAX, TRUE_KM)
v_data = v_true + rng.normal(0, 0.3, size=len(S_data))  # add measurement noise

# Fit
params, cov = curve_fit(
    michaelis_menten, S_data, v_data,
    p0=[5.0, 1.0],         # initial guess
    bounds=([0, 0], [np.inf, np.inf]),  # both params > 0
)
Vmax_fit, Km_fit = params
Vmax_se, Km_se = np.sqrt(np.diag(cov))

print(f"True    Vmax={TRUE_VMAX:.2f}, Km={TRUE_KM:.2f}")
print(f"Fitted  Vmax={Vmax_fit:.2f} ± {Vmax_se:.2f}, Km={Km_fit:.2f} ± {Km_se:.2f}")

In [ ]:
# Cell 6.2 — Negative log-likelihood minimization (normal distribution fitting)
# Given expression values, fit mean and std by MLE

from scipy.stats import norm

expr_data = rng.normal(loc=5.0, scale=1.5, size=200)

def neg_log_likelihood(theta: np.ndarray, data: np.ndarray) -> float:
    mu, log_sigma = theta
    sigma = np.exp(log_sigma)  # log-parameterize to enforce sigma > 0
    return -norm.logpdf(data, loc=mu, scale=sigma).sum()

result = minimize(
    neg_log_likelihood,
    x0=[4.0, np.log(2.0)],
    args=(expr_data,),
    method="L-BFGS-B",
)
mu_mle = result.x[0]
sigma_mle = np.exp(result.x[1])

print(f"True:    mu=5.00, sigma=1.50")
print(f"MLE:     mu={mu_mle:.3f}, sigma={sigma_mle:.3f}")
print(f"Sample:  mu={expr_data.mean():.3f}, sigma={expr_data.std():.3f}")
print(f"Converged: {result.success}")

In [ ]:
# Cell 6.3 — Root finding: at what substrate concentration is v = 80% of Vmax?
target = 0.8 * Vmax_fit

f = lambda S: michaelis_menten(S, Vmax_fit, Km_fit) - target
S_80 = brentq(f, a=0.01, b=1000.0)

print(f"v reaches 80% of Vmax at [S] = {S_80:.2f} µM")
print(f"Analytical check: 4 × Km = {4 * Km_fit:.2f} µM")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Michaelis-Menten fit
S_fine = np.linspace(0, 60, 300)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(S_data, v_data, color="steelblue", zorder=3, label="Data")
ax.plot(S_fine, michaelis_menten(S_fine, TRUE_VMAX, TRUE_KM),
        "--", color="gray", label=f"True: Vmax={TRUE_VMAX}, Km={TRUE_KM}")
ax.plot(S_fine, michaelis_menten(S_fine, Vmax_fit, Km_fit),
        color="tomato", label=f"Fit: Vmax={Vmax_fit:.2f}, Km={Km_fit:.2f}")
ax.axhline(0.8*Vmax_fit, color="green", lw=0.8, ls=":", alpha=0.7)
ax.axvline(S_80, color="green", lw=0.8, ls=":", alpha=0.7,
           label=f"[S] at 80% Vmax = {S_80:.1f} µM")
ax.set_xlabel("[S] (µM)")
ax.set_ylabel("v (µmol/min)")
ax.set_title("Michaelis-Menten curve fitting")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Fit the Hill equation $v = V_{\max} [S]^n / (K_{0.5}^n + [S]^n)$ to the same data
   and compare the fit quality using residual sum of squares.
2. What does the covariance matrix returned by `curve_fit` tell you? Extract the
   95% confidence intervals for $V_{\max}$ and $K_m$.
3. Replace `L-BFGS-B` with `Nelder-Mead` in Cell 6.2. How does the number of function
   evaluations compare?
4. Add a `fit_michaelis_menten(S, v)` function to `compbio_utils/stats.py` that returns
   a named tuple with `Vmax`, `Km`, `Vmax_se`, `Km_se`.

---
## Quiz — Active Recall

1. What is the Michaelis constant $K_m$ physically? What does a large $K_m$ mean?
2. Why do we minimize the *negative* log-likelihood instead of maximizing the log-likelihood?
3. Why is `log_sigma` (log-parameterization) used in Cell 6.2 instead of `sigma` directly?
4. What does `p0` in `curve_fit` control? What happens with a bad initial guess?
5. When would you use `differential_evolution` instead of `minimize`?

---
## Reflection

**Date completed:** ____________________

> *[Can you write the Michaelis-Menten curve_fit call from memory? Is fit_michaelis_menten in compbio_utils?]*

---
*Next: `12_scipy_odes_integration.ipynb`*